In [10]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\U1109200\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:

import pandas as pd
import sqlite3
import datetime
import re
import csv
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"SQLite version: {sqlite3.sqlite_version}")

try:
    df = pd.read_csv('customers-100.csv')
    print(f"Data loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print("\nFirst 3 rows:")
    print(df.head(3))
except Exception as e:
    print(f"Error loading data: {e}")

try:
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    df.to_sql('customers', conn, if_exists='replace', index=False)
    print("SQLite database created successfully!")
    print("Data loaded into 'customers' table")
    cursor.execute("SELECT COUNT(*) FROM customers")
    count = cursor.fetchone()[0]
    print(f"Total records in database: {count}")
except Exception as e:
    print(f"Error creating database: {e}")

test_results = []

def add_test_result(test_name, passed, details=""):
    test_results.append({
        'Test': test_name,
        'Status': 'PASS' if passed else 'FAIL',
        'Details': details
    })
    print(f"{'✓' if passed else '✗'} {test_name}: {'PASS' if passed else 'FAIL'}")
    if details:
        print(f"  Details: {details}")

print("=" * 60)
print("RUNNING DATA QUALITY TESTS")
print("=" * 60)

try:
    cursor.execute("""
        SELECT COUNT(*) as total_records,
               COUNT(DISTINCT [Customer Id]) as unique_ids
        FROM customers
    """)
    result = cursor.fetchone()
    total_records, unique_ids = result
    duplicates = total_records - unique_ids
    passed = duplicates == 0
    add_test_result("Test 1: Duplicate Customer IDs", passed, f"Found {duplicates} duplicate Customer IDs")
except Exception as e:
    add_test_result("Test 1: Duplicate Customer IDs", False, f"Error: {e}")

try:
    cursor.execute("""
        SELECT 
                   COUNT(*) as invalid_emails
        FROM 
                   customers 
        WHERE 
                   Email NOT LIKE '%@%' 
                    OR Email IS NULL 
                    OR Email = ''
    """)
    invalid_count = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM customers")
    total_count = cursor.fetchone()[0]
    passed = invalid_count == 0
    add_test_result("Test 2: Email Format Validation", passed, f"Found {invalid_count} invalid emails out of {total_count} records")
except Exception as e:
    add_test_result("Test 2: Email Format Validation", False, f"Error: {e}")

try:
    critical_fields = ['Customer Id', 'First Name', 'Last Name', 'Email', 'Country']
    null_counts = {}
    for field in critical_fields:
        cursor.execute(f"""
            SELECT COUNT(*) 
            FROM customers 
            WHERE [{field}] IS NULL OR [{field}] = ''
        """)
        null_counts[field] = cursor.fetchone()[0]
    total_nulls = sum(null_counts.values())
    passed = total_nulls == 0
    details = ", ".join([f"{field}: {count}" for field, count in null_counts.items() if count > 0])
    if not details:
        details = "No missing values found"
    add_test_result("Test 3: NULL/Missing Values Check", passed, details)
except Exception as e:
    add_test_result("Test 3: NULL/Missing Values Check", False, f"Error: {e}")

try:
    cursor.execute("""
        SELECT COUNT(*) as invalid_phones
        FROM customers 
        WHERE ([Phone 1] IS NULL OR [Phone 1] = '' OR LENGTH([Phone 1]) < 10)
          AND ([Phone 2] IS NULL OR [Phone 2] = '' OR LENGTH([Phone 2]) < 10)
    """)
    invalid_count = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM customers")
    total_count = cursor.fetchone()[0]
    passed = invalid_count == 0
    add_test_result("Test 4: Phone Number Format Validation", passed, f"Found {invalid_count} records with invalid phone numbers out of {total_count}")
except Exception as e:
    add_test_result("Test 4: Phone Number Format Validation", False, f"Error: {e}")

try:
    cursor.execute("""
        SELECT COUNT(*) as invalid_dates
        FROM customers 
        WHERE [Subscription Date] IS NULL 
           OR [Subscription Date] = ''
           OR date([Subscription Date]) IS NULL
    """)
    invalid_count = cursor.fetchone()[0]
    cursor.execute("SELECT COUNT(*) FROM customers")
    total_count = cursor.fetchone()[0]
    passed = invalid_count == 0
    add_test_result("Test 5: Subscription Date Validity", passed, f"Found {invalid_count} invalid dates out of {total_count} records")
except Exception as e:
    add_test_result("Test 5: Subscription Date Validity", False, f"Error: {e}")

try:
    cursor.execute("SELECT DISTINCT Country FROM customers WHERE Country IS NOT NULL")
    countries = [row[0] for row in cursor.fetchall()]
    invalid_countries = [c for c in countries if len(c.strip()) < 3]
    passed = len(invalid_countries) == 0
    add_test_result("Test 6: Country Name Validation", passed, f"Found {len(invalid_countries)} potentially invalid country names: {invalid_countries}")
except Exception as e:
    add_test_result("Test 6: Country Name Validation", False, f"Error: {e}")

try:
    cursor.execute("""
        SELECT Email, COUNT(*) as count
        FROM customers 
        WHERE Email IS NOT NULL AND Email != ''
        GROUP BY Email 
        HAVING COUNT(*) > 1
    """)
    duplicates = cursor.fetchall()
    passed = len(duplicates) == 0
    details = f"Found {len(duplicates)} duplicate email addresses"
    if duplicates:
        details += f": {[email for email, count in duplicates]}"
    add_test_result("Test 7: Duplicate Email Addresses", passed, details)
except Exception as e:
    add_test_result("Test 7: Duplicate Email Addresses", False, f"Error: {e}")

try:
    cursor.execute("""
        SELECT COUNT(*) as invalid_urls
        FROM customers 
        WHERE Website IS NOT NULL 
          AND Website != ''
          AND Website NOT LIKE 'http%'
    """)
    invalid_count = cursor.fetchone()[0]
    cursor.execute("""
        SELECT COUNT(*) 
        FROM customers 
        WHERE Website IS NOT NULL AND Website != ''
    """)
    total_urls = cursor.fetchone()[0]
    passed = invalid_count == 0
    add_test_result("Test 8: Website URL Format Validation", passed, f"Found {invalid_count} invalid URLs out of {total_urls} non-empty URLs")
except Exception as e:
    add_test_result("Test 8: Website URL Format Validation", False, f"Error: {e}")

try:
    required_fields = ['Customer Id', 'First Name', 'Last Name', 'Email']
    empty_counts = {}
    for field in required_fields:
        cursor.execute(f"""
            SELECT COUNT(*) 
            FROM customers 
            WHERE TRIM([{field}]) = ''
        """)
        empty_counts[field] = cursor.fetchone()[0]
    total_empty = sum(empty_counts.values())
    passed = total_empty == 0
    details = ", ".join([f"{field}: {count}" for field, count in empty_counts.items() if count > 0])
    if not details:
        details = "No empty strings found in required fields"
    add_test_result("Test 9: Empty Strings in Required Fields", passed, details)
except Exception as e:
    add_test_result("Test 9: Empty Strings in Required Fields", False, f"Error: {e}")

try:
    cursor.execute("SELECT COUNT(*) FROM customers")
    total_records = cursor.fetchone()[0]
    completeness_stats = {}
    for column in df.columns:
        if column != 'Index':
            cursor.execute(f"""
                SELECT COUNT(*) 
                FROM customers 
                WHERE [{column}] IS NOT NULL AND TRIM([{column}]) != ''
            """)
            non_empty = cursor.fetchone()[0]
            completeness_stats[column] = (non_empty / total_records) * 100
    critical_fields = ['Customer Id', 'First Name', 'Last Name', 'Email', 'Country']
    critical_completeness = [completeness_stats[field] for field in critical_fields]
    passed = all(comp >= 95.0 for comp in critical_completeness)
    avg_completeness = sum(completeness_stats.values()) / len(completeness_stats)
    details = f"Average completeness: {avg_completeness:.1f}%"
    add_test_result("Test 10: Data Completeness Summary", passed, details)
    print("\nDetailed Completeness Report:")
    for field, completeness in completeness_stats.items():
        print(f"  {field}: {completeness:.1f}%")
except Exception as e:
    add_test_result("Test 10: Data Completeness Summary", False, f"Error: {e}")

print("\n" + "=" * 60)
print("SQL QUERY EXAMPLES")
print("=" * 60)

print("\n1. Count customers by country:")
try:
    cursor.execute("""
        SELECT Country, COUNT(*) as customer_count
        FROM customers 
        GROUP BY Country 
        ORDER BY customer_count DESC
    """)
    results = cursor.fetchall()
    for country, count in results:
        print(f"  {country}: {count} customers")
except Exception as e:
    print(f"Error: {e}")

print("\n2. Customers subscribed in 2021:")
try:
    cursor.execute("""
        SELECT [First Name], [Last Name], [Subscription Date], Country
        FROM customers 
        WHERE strftime('%Y', [Subscription Date]) = '2021'
        ORDER BY [Subscription Date]
    """)
    results = cursor.fetchall()
    for fname, lname, sub_date, country in results:
        print(f"  {fname} {lname} - {sub_date} ({country})")
except Exception as e:
    print(f"Error: {e}")

print("\n3. Customers with potentially invalid emails:")
try:
    cursor.execute("""
        SELECT [First Name], [Last Name], Email
        FROM customers 
        WHERE Email NOT LIKE '%@%.%' 
           OR Email IS NULL 
           OR Email = ''
        ORDER BY [Last Name]
    """)
    results = cursor.fetchall()
    if results:
        for fname, lname, email in results:
            print(f"  {fname} {lname}: {email}")
    else:
        print("  No invalid emails found!")
except Exception as e:
    print(f"Error: {e}")

print("\n4. Customers grouped by subscription month (2021):")
try:
    cursor.execute("""
        SELECT strftime('%m', [Subscription Date]) as month,
               COUNT(*) as customer_count
        FROM customers 
        WHERE strftime('%Y', [Subscription Date]) = '2021'
        GROUP BY strftime('%m', [Subscription Date])
        ORDER BY month
    """)
    results = cursor.fetchall()
    month_names = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    for month_num, count in results:
        month_name = month_names[int(month_num)]
        print(f"  {month_name} 2021: {count} customers")
except Exception as e:
    print(f"Error: {e}")

print("\n5. Top 5 countries with most customers:")
try:
    cursor.execute("""
        SELECT Country, COUNT(*) as customer_count
        FROM customers 
        GROUP BY Country 
        ORDER BY customer_count DESC 
        LIMIT 5
    """)
    results = cursor.fetchall()
    for i, (country, count) in enumerate(results, 1):
        print(f"  {i}. {country}: {count} customers")
except Exception as e:
    print(f"Error: {e}")

print("\n" + "=" * 60)
print("TEST RESULTS SUMMARY")
print("=" * 60)

passed_tests = sum(1 for result in test_results if result['Status'] == 'PASS')
failed_tests = len(test_results) - passed_tests

print(f"\nTotal Tests Run: {len(test_results)}")
print(f"Passed: {passed_tests}")
print(f"Failed: {failed_tests}")
print(f"Success Rate: {(passed_tests/len(test_results)*100):.1f}%")

print("\nDetailed Results:")
print("-" * 60)
for result in test_results:
    status_symbol = "✓" if result['Status'] == 'PASS' else "✗"
    print(f"{status_symbol} {result['Test']}: {result['Status']}")
    if result['Details']:
        print(f"    {result['Details']}")

def export_test_results(filename='test_results.csv'):
    try:
        results_df = pd.DataFrame(test_results)
        results_df.to_csv(filename, index=False)
        print(f"\nTest results exported to {filename}")
        return True
    except Exception as e:
        print(f"Error exporting results: {e}")
        return False

export_success = export_test_results()

def create_summary_report():
    try:
        cursor.execute("SELECT COUNT(*) FROM customers")
        total_records = cursor.fetchone()[0]
        summary = {
            'Total Records': total_records,
            'Tests Run': len(test_results),
            'Tests Passed': sum(1 for r in test_results if r['Status'] == 'PASS'),
            'Tests Failed': sum(1 for r in test_results if r['Status'] == 'FAIL'),
            'Data Quality Score': f"{(sum(1 for r in test_results if r['Status'] == 'PASS')/len(test_results)*100):.1f}%"
        }
        print("\n" + "=" * 60)
        print("FINAL SUMMARY REPORT")
        print("=" * 60)
        for key, value in summary.items():
            print(f"{key}: {value}")
        return summary
    except Exception as e:
        print(f"Error creating summary: {e}")
        return None

summary_report = create_summary_report()

conn.close()
print(f"\nDatabase connection closed.")
print("Analysis complete!")


Libraries imported successfully!
Pandas version: 3.0.0
SQLite version: 3.50.4
Data loaded successfully!
Dataset shape: (100, 12)
Columns: ['Index', 'Customer Id', 'First Name', 'Last Name', 'Company', 'City', 'Country', 'Phone 1', 'Phone 2', 'Email', 'Subscription Date', 'Website']

First 3 rows:
   Index      Customer Id First Name Last Name          Company  \
0      1  DD37Cf93aecA6Dc     Sheryl    Baxter  Rasmussen Group   
1      2  1Ef7b82A4CAAD10    Preston    Lozano      Vega-Gentry   
2      3  6F94879bDAfE5a6        Roy     Berry    Murillo-Perry   

                City              Country          Phone 1  \
0       East Leonard                Chile     229.077.5154   
1  East Jimmychester             Djibouti       5153435776   
2      Isabelborough  Antigua and Barbuda  +1-539-402-0259   

               Phone 2                     Email Subscription Date  \
0     397.884.0519x718  zunigavanessa@smith.info        2020-08-24   
1     686-620-1820x944           vmata@colon

In [2]:
%%time

for i in range(10000000):
    pass

CPU times: total: 203 ms
Wall time: 209 ms


In [10]:
import sqlite3
import pandas as pd

# connect to sqlite database
conn = sqlite3.connect("customers.db")
cursor = conn.cursor()

# load csv file
df = pd.read_csv("customers-100.csv")

# load data into sqlite table
df.to_sql("customers", conn, if_exists="replace", index=False)

# ---- DATA QUALITY CHECKS ----
query = """
SELECT
  COUNT(*) AS total_rows,

  -- Completeness checks
  SUM(CASE WHEN "First Name" IS NULL OR "First Name" = '' THEN 1 ELSE 0 END) AS missing_first_name,
  SUM(CASE WHEN "Last Name"  IS NULL OR "Last Name"  = '' THEN 1 ELSE 0 END) AS missing_last_name,
  SUM(CASE WHEN "Email"      IS NULL OR "Email"      = '' THEN 1 ELSE 0 END) AS missing_email,
  SUM(CASE WHEN "Phone 1"    IS NULL OR "Phone 1"    = '' THEN 1 ELSE 0 END) AS missing_phone1,
  SUM(CASE WHEN "Phone 2"    IS NULL OR "Phone 2"    = '' THEN 1 ELSE 0 END) AS missing_phone2,

  -- Uniqueness checks
  (COUNT(*) - COUNT(DISTINCT "Customer Id")) AS duplicate_customer_id,
  (COUNT(*) - COUNT(DISTINCT "Email"))       AS duplicate_email,

  -- Validity checks
  SUM(CASE WHEN "Email" NOT LIKE '%@%' THEN 1 ELSE 0 END) AS invalid_email,
  SUM(CASE
        WHEN "Website" IS NULL OR "Website" = '' THEN 1
        WHEN "Website" NOT LIKE 'http%' AND "Website" NOT LIKE 'www%' THEN 1
        ELSE 0
      END) AS invalid_website,

  -- Timeliness check (works only if dates are in ISO 'YYYY-MM-DD' or a format SQLite can parse)
  SUM(CASE
        WHEN date("Subscription Date") IS NOT NULL AND date("Subscription Date") > date('now') THEN 1
        ELSE 0
      END) AS future_subscription_date

FROM customers;
"""
cursor.execute(query)
result = cursor.fetchall()
columns = [desc[0] for desc in cursor.description]

result_dict = dict(zip(columns, result[0]))

for k, v in result_dict.items():
    print(k, ":", v)


conn.close()

total_rows : 100
missing_first_name : 0
missing_last_name : 0
missing_email : 0
missing_phone1 : 0
missing_phone2 : 0
duplicate_customer_id : 0
duplicate_email : 0
invalid_email : 0
invalid_website : 0
future_subscription_date : 0


In [11]:
%%time

for i in range(10000000):
    pass

CPU times: total: 219 ms
Wall time: 191 ms
